
# Merge dual-line SRT (Primera línea SRT1, segunda línea SRT2 en amarillo)

Este notebook toma dos ficheros `.srt` y genera un `.srt` nuevo con:
- **Línea 1**: texto del **primer SRT** (se respetan sus timings).
- **Línea 2**: texto del **segundo SRT** en **amarillo** (`<font color="#FFFF00">…</font>`), alineado por solapamiento temporal o, si no hay, por el cue más cercano.

## Uso rápido
1. Ajusta las rutas en la **Celda 4** (`IN1`, `IN2`, `OUT`).
2. Ejecuta las celdas **en orden**.
3. Obtendrás el archivo mergeado en `OUT`.


In [1]:

from dataclasses import dataclass
from typing import List, Optional
from pathlib import Path
import re

SRT_TIME_RE = re.compile(
    r"(?P<h>\d{2}):(?P<m>\d{2}):(?P<s>\d{2}),(?P<ms>\d{3})\s*-->\s*"
    r"(?P<h2>\d{2}):(?P<m2>\d{2}):(?P<s2>\d{2}),(?P<ms2>\d{3})"
)

@dataclass
class Cue:
    idx: int
    start_ms: int
    end_ms: int
    text: str

def parse_time_to_ms(t: str) -> int:
    h, m, s_ms = t.split(":")
    s, ms = s_ms.split(",")
    return (int(h) * 3600 + int(m) * 60 + int(s)) * 1000 + int(ms)

def ms_to_time(ms: int) -> str:
    if ms < 0:
        ms = 0
    h = ms // 3600000
    ms %= 3600000
    m = ms // 60000
    ms %= 60000
    s = ms // 1000
    ms %= 1000
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

def parse_srt(path: str) -> List[Cue]:
    cues: List[Cue] = []
    text = Path(path).read_text(encoding="utf-8", errors="ignore")
    blocks = re.split(r"\n\s*\n", text.strip(), flags=re.MULTILINE)
    idx_counter = 1
    for block in blocks:
        lines = block.strip().splitlines()
        if not lines:
            continue
        # Detectar posible índice en primera línea
        time_line = None
        line_offset = 0
        if re.fullmatch(r"\d+", lines[0].strip()):
            line_offset = 1
        if line_offset < len(lines):
            cand = lines[line_offset].strip()
            if "-->" in cand:
                time_line = cand
        if not time_line:
            continue
        m = SRT_TIME_RE.search(time_line)
        if not m:
            continue
        start_ms = parse_time_to_ms(f"{m['h']}:{m['m']}:{m['s']},{m['ms']}")
        end_ms = parse_time_to_ms(f"{m['h2']}:{m['m2']}:{m['s2']},{m['ms2']}")
        text_lines = lines[line_offset + 1:]
        txt = "\n".join(text_lines).strip()
        cues.append(Cue(idx=idx_counter, start_ms=start_ms, end_ms=end_ms, text=txt))
        idx_counter += 1
    return cues

def overlap_ms(a_start: int, a_end: int, b_start: int, b_end: int) -> int:
    return max(0, min(a_end, b_end) - max(a_start, b_start))

def single_line(text: str) -> str:
    parts = [ln.strip() for ln in text.splitlines() if ln.strip()]
    return re.sub(r"\s+", " ", " ".join(parts))

def merge_dual_srt(
    srt1_path: str,
    srt2_path: str,
    out_path: str,
    second_line_hex: str = "#FFFF00",
    collapse_lines: bool = True,
    min_overlap_ms: int = 500,
    min_overlap_ratio: float = 0.25,
    nearest_gap_ms: int = 1000
) -> int:
    """
    Genera un SRT con timings de srt1 y dos líneas por cue:
      - Línea 1: texto de srt1
      - Línea 2: texto de srt2 en color (hex), si hay solapamiento suficiente (o cue más cercano)
    Parámetros:
      - second_line_hex: color de la 2ª línea (por defecto amarillo).
      - collapse_lines: si True, colapsa saltos de línea en una sola línea por idioma.
      - min_overlap_ms / min_overlap_ratio: umbrales para considerar solapamiento.
      - nearest_gap_ms: si no hay solapamiento, acepta el cue srt2 más cercano dentro de este umbral.
    Devuelve: número de cues escritos.
    """
    cues1 = parse_srt(srt1_path)
    cues2 = parse_srt(srt2_path)

    out_lines = []
    count = 0

    for i, c1 in enumerate(cues1, 1):
        dur1 = max(1, c1.end_ms - c1.start_ms)
        threshold = max(min_overlap_ms, int(dur1 * min_overlap_ratio))

        # Buscar cues srt2 que solapan con c1
        overlapped = []
        for c2 in cues2:
            ov = overlap_ms(c1.start_ms, c1.end_ms, c2.start_ms, c2.end_ms)
            if ov >= threshold:
                overlapped.append(c2)

        # Si no hay solape, elegir el más cercano (si está dentro del umbral)
        if not overlapped:
            nearest: Optional[Cue] = None
            best_gap = 10**9
            for c2 in cues2:
                if c2.end_ms < c1.start_ms:
                    gap = c1.start_ms - c2.end_ms
                elif c2.start_ms > c1.end_ms:
                    gap = c2.start_ms - c1.end_ms
                else:
                    gap = 0
                if gap < best_gap:
                    best_gap = gap
                    nearest = c2
            if nearest and best_gap <= nearest_gap_ms:
                overlapped = [nearest]

        # Preparar líneas
        line1 = single_line(c1.text) if collapse_lines else c1.text
        if overlapped:
            joined = " ".join([c2.text for c2 in overlapped]) if collapse_lines else "\n".join([c2.text for c2 in overlapped])
            line2_txt = single_line(joined) if collapse_lines else joined
        else:
            line2_txt = ""

        # Escribir cue
        out_lines.append(str(i))
        out_lines.append(f"{ms_to_time(c1.start_ms)} --> {ms_to_time(c1.end_ms)}")
        out_lines.append(line1 if line1 else "")
        if line2_txt:
            out_lines.append(f'<font color="{second_line_hex}">{line2_txt}</font>')
        out_lines.append("")
        count += 1

    Path(out_path).write_text("\n".join(out_lines), encoding="utf-8")
    return count


In [2]:

# === Configura aquí tus rutas y preferencias ===
IN1 = "output_es.srt"   # SRT principal (timings)
IN2 = "output_zh-hans.srt"             # SRT secundario (segunda línea)
OUT = "output_final.srt"

SECOND_LINE_COLOR = "#FFFF00"  # Amarillo
COLLAPSE_LINES = True          # True: colapsar saltos de línea a una sola línea por idioma
MIN_OVERLAP_MS = 500           # Mínimo solape en ms para considerar un cue coincidente
MIN_OVERLAP_RATIO = 0.25       # % del tiempo del cue1 que debe solapar como mínimo
NEAREST_GAP_MS = 1000          # Si no hay solape, aceptar cue srt2 más cercano dentro de este umbral


In [3]:

n = merge_dual_srt(
    IN1, IN2, OUT,
    second_line_hex=SECOND_LINE_COLOR,
    collapse_lines=COLLAPSE_LINES,
    min_overlap_ms=MIN_OVERLAP_MS,
    min_overlap_ratio=MIN_OVERLAP_RATIO,
    nearest_gap_ms=NEAREST_GAP_MS
)
print(f"Escribí {n} cues en: {OUT}")


Escribí 296 cues en: output_final.srt


In [4]:

# (Opcional) Vista previa de las primeras 10 entradas del SRT resultante
from itertools import islice

with open(OUT, "r", encoding="utf-8") as f:
    print("".join(list(islice(f, 60))))


1
00:00:03,656 --> 00:00:05,656
¡Gracias!
<font color="#FFFF00">谢谢!</font>

2
00:00:28,104 --> 00:00:30,864
Me viene Pilar que a ver si le puedo llevar en su coche al banco.
<font color="#FFFF00">皮拉尔要来了 我可以开车载她去银行</font>

3
00:00:31,544 --> 00:00:33,464
Una faena. Me habían quitado todos los puntos.
<font color="#FFFF00">有一件事,他们把我所有的分数都拿走了</font>

4
00:00:34,344 --> 00:00:35,484
Nada, no se tarda nada.
<font color="#FFFF00">没什么,不会花很久的</font>

5
00:00:36,360 --> 00:00:37,560
Un momentito.
<font color="#FFFF00">请稍等一下</font>

6
00:00:37,640 --> 00:00:39,440
Un momentito para esta...
<font color="#FFFF00">一分钟,如果那...</font>

7
00:00:39,976 --> 00:00:41,356
Es ir a Cobo Calleja y volver.
<font color="#FFFF00">它要去科博卡梅雅 回来。</font>

8
00:00:41,876 --> 00:00:43,816
Me habían dicho que allí daban los préstamos sin pensar.
<font color="#FFFF00">他们告诉我,他们发放贷款时没有考虑。</font>

9
00:00:44,216 --> 00:00:45,516
No sé quién le habría dicho eso.
<font color="#FFFF00">我不知道谁会告诉他这些</font>

10
00:00:46,952 --> 0